# NSEMI Script 1c — ISM Verification 2026 (May 1, 2026)

**Purpose:** Verify the count and identity of India's ISM-approved semiconductor manufacturing
projects against authoritative PIB press releases. Build new RQ2 reference datasets without
modifying existing files.

**Why this notebook exists:**

The interim report (April 29, 2026) operated on an assumption of 3 ISM-approved states
(Gujarat, Assam, Uttar Pradesh). Verification against PIB press releases on May 1, 2026
revealed the canonical count is **10 projects across 6 states** as of the analysis cutoff,
with cumulative committed investment of approximately ₹1.60 lakh crore.

**Six host states:** Gujarat, Assam, Uttar Pradesh, Odisha, Punjab, Andhra Pradesh.

**Authoritative source:** PIB press release PRID 2155456 (12 Aug 2025), which states verbatim:
*"With these four more approvals today, total approved projects under ISM reaches to 10
with cumulative investments of around Rs.1.60 lakh crore in 6 states."*

**Scope of this notebook:**

- Downloads 5 primary-source PIB press releases as audit-trail PDFs.
- Builds canonical ISM_APPROVED_PROJECTS list (10 projects) with full provenance.
- Aggregates to state-level summary (6 ISM states + 5 tier-1 policy-only states).
- Adds 4-tier maturity ordinal: 4=Operational, 3=Under Construction, 2=Pre-Construction, 1=State Policy Only.
- Saves NEW files alongside existing files (does NOT modify rq2_pfc_atc_losses.csv or
  rq2_semiconductor_facility_locations.csv).

**Files this notebook produces:**
- `raw/ism_pib_releases/PIB_PRID_*.pdf` (5 PDFs)
- `raw/ism_pib_releases/ism_pib_provenance.json` (audit log)
- `cleaned/rq2_ism_approved_projects_n10_verified_2026_05.csv` (10 projects × 11 cols)
- `cleaned/rq2_ism_approved_states_n6_verified_2026_05.csv` (6 ISM + 5 policy-only + tier-0 rest)

**Files this notebook does NOT modify:**
- `cleaned/rq2_pfc_atc_losses.csv` — preserved unchanged
- `cleaned/rq2_semiconductor_facility_locations.csv` — preserved unchanged
- `NSEMI_Script1_Data_Extraction.ipynb` — preserved unchanged
- `NSEMI_Script3_Data_Cleaning.ipynb` — preserved unchanged

## Cell 1 — Setup, paths, and authoritative source registry

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import requests
import json
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

DRIVE_BASE = Path('/content/drive/MyDrive/Walsh_Masters/Term-2_Capstone')
RAW = DRIVE_BASE / 'raw'
CLEANED = DRIVE_BASE / 'cleaned'
ISM_DIR = RAW / 'ism_pib_releases'
ISM_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now().isoformat()

print("=" * 70)
print("NSEMI Script 1c — ISM Verification 2026")
print("=" * 70)

# Registry of canonical PIB sources for ISM project approvals
PIB_SOURCES = [
    {
        'prid': '1983128',
        'date': '2023-08-30',
        'event': 'Micron Sanand status update (Cabinet approval was June 2023)',
        'url': 'https://www.pib.gov.in/PressReleasePage.aspx?PRID=1983128',
        'projects_announced': 1,
        'covers': ['Micron Sanand ATMP'],
    },
    {
        'prid': '2010132',
        'date': '2024-02-29',
        'event': 'Cabinet approves 3 semiconductor units (Tata Dholera + Tata Assam + CG Semi)',
        'url': 'https://www.pib.gov.in/PressReleasePage.aspx?PRID=2010132',
        'projects_announced': 3,
        'covers': ['Tata Electronics PSMC Fab Dholera', 'Tata TSAT Jagiroad', 'CG Semi Sanand'],
    },
    {
        'prid': '2050859',
        'date': '2024-09-02',
        'event': 'Cabinet approves Kaynes Semicon Sanand (5th ISM unit)',
        'url': 'https://www.pib.gov.in/PressReleaseIframePage.aspx?PRID=2050859',
        'projects_announced': 1,
        'covers': ['Kaynes Semicon Sanand'],
    },
    {
        'prid': '2128605',
        'date': '2025-05-14',
        'event': 'Cabinet approves HCL-Foxconn JV in Uttar Pradesh (6th ISM unit)',
        'url': 'https://www.pib.gov.in/PressReleasePage.aspx?PRID=2128605',
        'projects_announced': 1,
        'covers': ['HCL-Foxconn India Chip Pvt Ltd Jewar'],
    },
    {
        'prid': '2155456',
        'date': '2025-08-12',
        'event': 'Cabinet approves 4 units in Odisha + Punjab + AP (units 7-10; total reaches 10 across 6 states)',
        'url': 'https://www.pib.gov.in/PressReleasePage.aspx?PRID=2155456',
        'projects_announced': 4,
        'covers': ['SiCSem Odisha', '3D Glass Solutions HIPSPL Odisha', 'CDIL Punjab', 'ASIP AP'],
    },
]

print(f"\nPIB source registry ({len(PIB_SOURCES)} press releases):")
for src in PIB_SOURCES:
    print(f"  [{src['date']}] PRID {src['prid']}: {src['event']}")

print(f"\n--- Setup complete ---")
print(f"Output directory: {ISM_DIR}")

Mounted at /content/drive
NSEMI Script 1c — ISM Verification 2026

PIB source registry (5 press releases):
  [2023-08-30] PRID 1983128: Micron Sanand status update (Cabinet approval was June 2023)
  [2024-02-29] PRID 2010132: Cabinet approves 3 semiconductor units (Tata Dholera + Tata Assam + CG Semi)
  [2024-09-02] PRID 2050859: Cabinet approves Kaynes Semicon Sanand (5th ISM unit)
  [2025-05-14] PRID 2128605: Cabinet approves HCL-Foxconn JV in Uttar Pradesh (6th ISM unit)
  [2025-08-12] PRID 2155456: Cabinet approves 4 units in Odisha + Punjab + AP (units 7-10; total reaches 10 across 6 states)

--- Setup complete ---
Output directory: /content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/raw/ism_pib_releases


In [3]:
print('--- Installing weasyprint for HTML to PDF conversion ---')
!pip install weasyprint
from weasyprint import HTML
print('--- Installation complete ---')

--- Installing weasyprint for HTML to PDF conversion ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.4/829.4 kB 39.4 MB/s eta 0:00:00
  Attempting uninstall: tinycss2
    Found existing installation: tinycss2 1.4.0
    Uninstalling tinycss2-1.4.0:
      Successfully uninstalled tinycss2-1.4.0
--- Installation complete ---


## Cell 2 — Download PIB press releases for audit trail

Downloads each PIB URL as HTML (PIB press release pages are dynamic ASP.NET pages, not static PDFs).
Saves as HTML for archival; can be converted to PDF later if needed for printable inclusion.
Validates each download with size threshold and content sanity check.

In [9]:
print("=" * 70)
print("Cell 2 — Verify and rename manually-uploaded PIB press releases")
print("=" * 70)

# Manual download approach (May 2026): PIB pages are protected by Akamai bot detection
# which blocks headless browsers. Manual browser print-to-PDF is the cleanest method.
# This cell expects the 5 PIB PDFs to be uploaded to ISM_DIR first, then identifies
# and renames them to the canonical PIB_PRID_<id>.pdf format.

import subprocess
import sys
import re
import json
from datetime import datetime

# Ensure pypdf is available for PRID identification
try:
    from pypdf import PdfReader
except ImportError:
    print("Installing pypdf...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'pypdf', '--quiet'], check=True)
    from pypdf import PdfReader

# Expected PRIDs with content keywords for verification
EXPECTED_PRIDS = {
    '1983128': {
        'date': '2023-12-06',
        'topic': 'Micron Sanand status update',
        'keywords': ['Micron', 'Sanand', '22,516', 'fast track'],
    },
    '2010132': {
        'date': '2024-02-29',
        'topic': 'Tata Dholera + Tata Assam + CG Semi (3 units)',
        'keywords': ['Tata Electronics', 'Dholera', 'PSMC', 'three more'],
    },
    '2050859': {
        'date': '2024-09-02',
        'topic': 'Kaynes Semicon Sanand',
        'keywords': ['Kaynes Semicon', '3,300 crore', '60 Lakh'],
    },
    '2128605': {
        'date': '2025-05-14',
        'topic': 'HCL-Foxconn JV in Uttar Pradesh',
        'keywords': ['HCL', 'Foxconn', 'Uttar Pradesh', 'Jewar'],
    },
    '2155456': {
        'date': '2025-08-12',
        'topic': 'Odisha + Punjab + Andhra Pradesh (4 units, total reaches 10)',
        'keywords': ['Odisha', 'Punjab', 'Andhra Pradesh', 'SiCSem', '4,600 crore'],
    },
}

# Step 1: Inventory current PDFs in ISM_DIR
print(f"\nStep 1: Inventory of PDFs in {ISM_DIR}")
all_pdfs = sorted(ISM_DIR.glob('*.pdf'))
already_canonical = [p for p in all_pdfs if re.match(r'PIB_PRID_\d+\.pdf$', p.name)]
needs_rename = [p for p in all_pdfs if p not in already_canonical]

print(f"  Total PDFs found: {len(all_pdfs)}")
print(f"  Already canonical (PIB_PRID_<id>.pdf): {len(already_canonical)}")
print(f"  Needs rename / identification: {len(needs_rename)}")

# Step 2: Identify and rename PDFs
print(f"\nStep 2: Identify and rename uploaded PDFs")

rename_log = []

# First, log already-canonical files
for pdf_path in already_canonical:
    prid = pdf_path.stem.replace('PIB_PRID_', '')
    expected = EXPECTED_PRIDS.get(prid, {})
    print(f"\n  ✓ {pdf_path.name}: already canonical")
    print(f"    PRID {prid} — {expected.get('topic', 'unknown')}")
    rename_log.append({
        'final_name': pdf_path.name,
        'prid': prid,
        'topic': expected.get('topic', 'unknown'),
        'date': expected.get('date'),
        'size_kb': round(pdf_path.stat().st_size / 1024, 1),
        'status': 'ALREADY_CANONICAL',
        'identified_by': 'filename',
    })

# Then identify and rename non-canonical files
for pdf_path in needs_rename:
    print(f"\n  Inspecting: {pdf_path.name}")

    try:
        reader = PdfReader(str(pdf_path))
        first_page_text = reader.pages[0].extract_text()

        # Method A: Extract PRID from embedded URL
        prid_match = re.search(r'PRID=(\d+)', first_page_text)
        method = None
        prid = None

        if prid_match:
            prid = prid_match.group(1)
            method = 'PRID_from_URL'
        else:
            # Method B: Extract from "Release ID" line
            release_match = re.search(r'Release ID:\s*(\d+)', first_page_text)
            if release_match:
                prid = release_match.group(1)
                method = 'PRID_from_Release_ID'
            else:
                # Method C: Match by content keywords
                best_prid = None
                best_score = 0
                for test_prid, info in EXPECTED_PRIDS.items():
                    score = sum(1 for kw in info['keywords'] if kw.lower() in first_page_text.lower())
                    if score > best_score:
                        best_score = score
                        best_prid = test_prid
                if best_score >= 2:
                    prid = best_prid
                    method = f'content_keywords_match_{best_score}'

        if not prid:
            print(f"    ⚠ Could not identify PRID — skipping")
            rename_log.append({
                'original_name': pdf_path.name,
                'prid': None,
                'status': 'UNIDENTIFIED',
            })
            continue

        # Verify PRID is in expected set
        if prid not in EXPECTED_PRIDS:
            print(f"    ⚠ PRID {prid} not in expected set ({list(EXPECTED_PRIDS.keys())})")
            rename_log.append({
                'original_name': pdf_path.name,
                'prid': prid,
                'status': 'UNEXPECTED_PRID',
                'identified_by': method,
            })
            continue

        # Verify content keywords match
        expected = EXPECTED_PRIDS[prid]
        keyword_matches = sum(1 for kw in expected['keywords'] if kw.lower() in first_page_text.lower())

        # Rename
        target_path = ISM_DIR / f"PIB_PRID_{prid}.pdf"
        if target_path.exists() and target_path != pdf_path:
            print(f"    ⚠ Target {target_path.name} already exists — skipping rename")
            rename_log.append({
                'original_name': pdf_path.name,
                'prid': prid,
                'status': 'TARGET_EXISTS',
                'identified_by': method,
            })
            continue

        pdf_path.rename(target_path)
        size_kb = target_path.stat().st_size / 1024
        print(f"    ✓ PRID {prid} (matched {keyword_matches}/{len(expected['keywords'])} keywords via {method})")
        print(f"    ✓ Renamed → PIB_PRID_{prid}.pdf ({size_kb:.0f} KB)")
        print(f"    Topic: {expected['topic']}")

        rename_log.append({
            'original_name': pdf_path.name,
            'final_name': target_path.name,
            'prid': prid,
            'topic': expected['topic'],
            'date': expected['date'],
            'size_kb': round(size_kb, 1),
            'identified_by': method,
            'keyword_match_score': f"{keyword_matches}/{len(expected['keywords'])}",
            'status': 'RENAMED',
        })

    except Exception as e:
        print(f"    ✗ Error: {type(e).__name__}: {str(e)[:100]}")
        rename_log.append({
            'original_name': pdf_path.name,
            'status': f'ERROR_{type(e).__name__}',
            'error': str(e)[:200],
        })

# Step 3: Final coverage check
print(f"\nStep 3: Coverage check")
final_pdfs = sorted(ISM_DIR.glob('PIB_PRID_*.pdf'))
prids_present = {p.stem.replace('PIB_PRID_', '') for p in final_pdfs}
prids_expected = set(EXPECTED_PRIDS.keys())
prids_missing = prids_expected - prids_present

print(f"  Expected PRIDs: {sorted(prids_expected)}")
print(f"  Present PRIDs:  {sorted(prids_present)}")
if prids_missing:
    print(f"  ⚠ MISSING:      {sorted(prids_missing)}")
    for prid in sorted(prids_missing):
        info = EXPECTED_PRIDS[prid]
        print(f"     PRID {prid} ({info['date']}): {info['topic']}")
        print(f"     Source URL: https://www.pib.gov.in/PressReleasePage.aspx?PRID={prid}")
else:
    print(f"  ✓ All 5 expected PRIDs present")

# Step 4: Save provenance log
prov_path = ISM_DIR / 'ism_pib_provenance.json'
prov_data = {
    'data_source': 'PIB_press_releases_ISM_approvals',
    'collection_method': 'Manual browser print-to-PDF (Akamai bot protection blocks automated downloads)',
    'collection_date': '2026-05-02',
    'consolidation_timestamp': datetime.now().isoformat(),
    'rendering_engine': 'Browser native print (Chrome/Firefox Save as PDF)',
    'canonical_summary': {
        'total_ism_approved_projects': 10,
        'total_ism_approved_states': 6,
        'cumulative_investment_inr_crore': 159725,
        'cumulative_investment_usd_billion_approx': 18.2,
        'authoritative_count_source': 'PIB PRID 2155456 (12 Aug 2025)',
        'analysis_cutoff_date': '2026-05-01',
    },
    'sources': [
        {
            'prid': prid,
            'date': info['date'],
            'topic': info['topic'],
            'url': f"https://www.pib.gov.in/PressReleasePage.aspx?PRID={prid}",
            'local_file': f"PIB_PRID_{prid}.pdf",
            'present': prid in prids_present,
        }
        for prid, info in EXPECTED_PRIDS.items()
    ],
    'rename_log': rename_log,
    'completion_status': 'COMPLETE' if not prids_missing else 'PARTIAL',
}

with open(prov_path, 'w') as f:
    json.dump(prov_data, f, indent=2)

print(f"\n--- Files in {ISM_DIR.name}/ ---")
for f in sorted(ISM_DIR.glob('*')):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}: {size_kb:.0f} KB")

print(f"\n✓ Provenance log: {prov_path.name}")
print(f"✓ Status: {'COMPLETE — all 5 PIB sources archived' if not prids_missing else f'PARTIAL — {len(prids_missing)} PRIDs missing'}")

Cell 2 — Verify and rename manually-uploaded PIB press releases
Installing pypdf...

Step 1: Inventory of PDFs in /content/drive/MyDrive/Walsh_Masters/Term-2_Capstone/raw/ism_pib_releases
  Total PDFs found: 5
  Already canonical (PIB_PRID_<id>.pdf): 0
  Needs rename / identification: 5

Step 2: Identify and rename uploaded PDFs

  Inspecting: Press Release Page _ Press Information Bureau.pdf
    ✓ PRID 1983128 (matched 4/4 keywords via PRID_from_URL)
    ✓ Renamed → PIB_PRID_1983128.pdf (116 KB)
    Topic: Micron Sanand status update

  Inspecting: Press Release Page _ Press Information Bureau_2.pdf
    ✓ PRID 2010132 (matched 4/4 keywords via PRID_from_URL)
    ✓ Renamed → PIB_PRID_2010132.pdf (151 KB)
    Topic: Tata Dholera + Tata Assam + CG Semi (3 units)

  Inspecting: Press Release Page _ Press Information Bureau_4.pdf
    ✓ PRID 2128605 (matched 3/4 keywords via PRID_from_URL)
    ✓ Renamed → PIB_PRID_2128605.pdf (119 KB)
    Topic: HCL-Foxconn JV in Uttar Pradesh

  Inspecting

## Cell 3 — Build canonical ISM_APPROVED_PROJECTS dataset

Hardcodes the 10 ISM-approved projects with full attribution to PIB sources.
Each row carries: state, project name, type, investment INR cr, cabinet approval date,
operational status as of 2026-05-01, lead company, and source PIB URL.

**Excluded by design:** SCL Mohali (₹4,500 cr modernization, Nov 2025) is administered
separately by MeitY and is NOT counted in the 10-project ISM portfolio per PIB convention.
Including it would not change the state count (Punjab is already counted for CDIL) but
would inflate the ₹1.60 lakh crore cumulative figure.

In [10]:
print("=" * 70)
print("Cell 3 — Build canonical ISM_APPROVED_PROJECTS dataset")
print("=" * 70)

ISM_APPROVED_PROJECTS = [
    # === Approved June 2023 (1st ISM unit) — PIB PRID 1983128 ===
    {
        'project_id': 'ISM_001',
        'state_name': 'Gujarat',
        'project_name': 'Micron Semiconductor ATMP (Sanand)',
        'project_type': 'ATMP',
        'investment_inr_cr': 22516,
        'cabinet_approval_date': '2023-06-15',
        'status_2026_05_01': 'Operational',
        'lead_company': 'Micron Technology',
        'subsidy_pct_centre': 50,
        'subsidy_pct_state': 20,
        'pib_prid': '1983128',
        'pib_url': 'https://www.pib.gov.in/PressReleasePage.aspx?PRID=1983128',
        'notes': 'First ISM-approved unit; commercial production began 28 Feb 2026',
    },
    # === Approved Feb 2024 — PIB PRID 2010132 ===
    {
        'project_id': 'ISM_002',
        'state_name': 'Gujarat',
        'project_name': 'Tata Electronics PSMC Fab (Dholera)',
        'project_type': 'Fab',
        'investment_inr_cr': 91000,
        'cabinet_approval_date': '2024-02-29',
        'status_2026_05_01': 'Under_Construction',
        'lead_company': 'Tata Electronics + PSMC (Taiwan)',
        'subsidy_pct_centre': 50,
        'subsidy_pct_state': 20,
        'pib_prid': '2010132',
        'pib_url': 'https://www.pib.gov.in/PressReleasePage.aspx?PRID=2010132',
        'notes': "India's first commercial silicon CMOS fab; 28-110nm logic and power; "
                 "first chip targeted late 2026",
    },
    {
        'project_id': 'ISM_003',
        'state_name': 'Assam',
        'project_name': 'Tata Semiconductor Assembly & Test (TSAT, Jagiroad)',
        'project_type': 'OSAT_ATMP',
        'investment_inr_cr': 27000,
        'cabinet_approval_date': '2024-02-29',
        'status_2026_05_01': 'Under_Construction',
        'lead_company': 'Tata Electronics',
        'subsidy_pct_centre': 50,
        'subsidy_pct_state': 40,
        'pib_prid': '2010132',
        'pib_url': 'https://www.pib.gov.in/PressReleasePage.aspx?PRID=2010132',
        'notes': "Northeast India's first semiconductor unit; flip-chip + ISIP packaging",
    },
    {
        'project_id': 'ISM_004',
        'state_name': 'Gujarat',
        'project_name': 'CG Semi (CG Power + Renesas + Stars Microelectronics)',
        'project_type': 'OSAT_ATMP',
        'investment_inr_cr': 7600,
        'cabinet_approval_date': '2024-02-29',
        'status_2026_05_01': 'Pilot_Operational',
        'lead_company': 'CG Power',
        'subsidy_pct_centre': 50,
        'subsidy_pct_state': 40,
        'pib_prid': '2010132',
        'pib_url': 'https://www.pib.gov.in/PressReleasePage.aspx?PRID=2010132',
        'notes': 'G1 pilot inaugurated 28 Aug 2025; commercial production targeted 2026',
    },
    # === Approved Sep 2024 — PIB PRID 2050859 ===
    {
        'project_id': 'ISM_005',
        'state_name': 'Gujarat',
        'project_name': 'Kaynes Semicon (Sanand)',
        'project_type': 'OSAT',
        'investment_inr_cr': 3307,
        'cabinet_approval_date': '2024-09-02',
        'status_2026_05_01': 'Operational',
        'lead_company': 'Kaynes Technology',
        'subsidy_pct_centre': 50,
        'subsidy_pct_state': 40,
        'pib_prid': '2050859',
        'pib_url': 'https://www.pib.gov.in/PressReleaseIframePage.aspx?PRID=2050859',
        'notes': 'Inaugurated 31 March 2026; capacity 60 lakh chips/day',
    },
    # === Approved May 2025 — PIB PRID 2128605 ===
    {
        'project_id': 'ISM_006',
        'state_name': 'Uttar Pradesh',
        'project_name': 'India Chip Pvt Ltd (HCL-Foxconn JV, Jewar)',
        'project_type': 'OSAT',
        'investment_inr_cr': 3706,
        'cabinet_approval_date': '2025-05-14',
        'status_2026_05_01': 'Under_Construction',
        'lead_company': 'HCL Group + Foxconn',
        'subsidy_pct_centre': 50,
        'subsidy_pct_state': 100,  # UP policy ceiling — up to 100% of central incentive
        'pib_prid': '2128605',
        'pib_url': 'https://www.pib.gov.in/PressReleasePage.aspx?PRID=2128605',
        'notes': 'Display driver chips; 20K wafers/month; groundbreaking 21 Feb 2026',
    },
    # === Approved Aug 2025 — PIB PRID 2155456 ===
    {
        'project_id': 'ISM_007',
        'state_name': 'Odisha',
        'project_name': 'SiCSem Compound Semiconductor Fab + ATMP (Bhubaneswar)',
        'project_type': 'Compound_SiC_Fab_ATMP',
        'investment_inr_cr': 2067,
        'cabinet_approval_date': '2025-08-12',
        'status_2026_05_01': 'Under_Construction',
        'lead_company': 'SiCSem (Archean Chemicals + Clas-SiC UK)',
        'subsidy_pct_centre': 50,
        'subsidy_pct_state': 30,
        'pib_prid': '2155456',
        'pib_url': 'https://www.pib.gov.in/PressReleasePage.aspx?PRID=2155456',
        'notes': "India's first commercial SiC fab; 60K SiC wafers/yr; "
                 'groundbreaking 1 Nov 2025',
    },
    {
        'project_id': 'ISM_008',
        'state_name': 'Odisha',
        'project_name': '3D Glass Solutions / HIPSPL Advanced Packaging (Bhubaneswar)',
        'project_type': '3D_Glass_Packaging',
        'investment_inr_cr': 1944,
        'cabinet_approval_date': '2025-08-12',
        'status_2026_05_01': 'Under_Construction',
        'lead_company': '3D Glass Solutions Inc. + HIPSPL',
        'subsidy_pct_centre': 50,
        'subsidy_pct_state': 30,
        'pib_prid': '2155456',
        'pib_url': 'https://www.pib.gov.in/PressReleasePage.aspx?PRID=2155456',
        'notes': "India's first 3D glass-substrate packaging unit; "
                 'groundbreaking 19 Apr 2026',
    },
    {
        'project_id': 'ISM_009',
        'state_name': 'Punjab',
        'project_name': 'CDIL Mohali Brownfield Expansion',
        'project_type': 'Discrete_Semiconductors',
        'investment_inr_cr': 117,
        'cabinet_approval_date': '2025-08-12',
        'status_2026_05_01': 'Approved',
        'lead_company': 'Continental Device India Ltd (CDIL)',
        'subsidy_pct_centre': 50,
        'subsidy_pct_state': 0,
        'pib_prid': '2155456',
        'pib_url': 'https://www.pib.gov.in/PressReleasePage.aspx?PRID=2155456',
        'notes': 'Si/SiC MOSFETs, IGBTs, Schottky diodes; brownfield (existing facility expansion)',
    },
    {
        'project_id': 'ISM_010',
        'state_name': 'Andhra Pradesh',
        'project_name': 'ASIP Technologies + APACT Co. (South Korea)',
        'project_type': 'OSAT_SiP',
        'investment_inr_cr': 468,
        'cabinet_approval_date': '2025-08-12',
        'status_2026_05_01': 'Approved',
        'lead_company': 'ASIP Technologies + APACT (Korea)',
        'subsidy_pct_centre': 50,
        'subsidy_pct_state': 60,
        'pib_prid': '2155456',
        'pib_url': 'https://www.pib.gov.in/PressReleasePage.aspx?PRID=2155456',
        'notes': '96M units/yr; mobile, set-top box, automotive applications',
    },
]

projects_df = pd.DataFrame(ISM_APPROVED_PROJECTS)
projects_df['data_source'] = 'PIB_press_releases_ISM'
projects_df['retrieval_date'] = '2026-05-01'

# Save
out_projects = CLEANED / 'rq2_ism_approved_projects_n10_verified_2026_05.csv'
projects_df.to_csv(out_projects, index=False)

print(f"\n✓ Saved: {out_projects.name}")
print(f"  Shape: {projects_df.shape}")
print(f"  Total ISM-approved projects: {len(projects_df)}")
print(f"  Cumulative investment: ₹{projects_df['investment_inr_cr'].sum():,} crore "
      f"(~US${projects_df['investment_inr_cr'].sum()/8800:.1f}B at ~₹88/USD)")

print(f"\n--- Projects by state ---")
state_counts = projects_df.groupby('state_name').size().sort_values(ascending=False)
for state, count in state_counts.items():
    inv = projects_df[projects_df['state_name'] == state]['investment_inr_cr'].sum()
    print(f"  {state}: {count} project(s), ₹{inv:,} cr")

print(f"\n--- Projects by status (as of 2026-05-01) ---")
status_counts = projects_df.groupby('status_2026_05_01').size()
for status, count in status_counts.items():
    print(f"  {status}: {count}")

Cell 3 — Build canonical ISM_APPROVED_PROJECTS dataset

✓ Saved: rq2_ism_approved_projects_n10_verified_2026_05.csv
  Shape: (10, 15)
  Total ISM-approved projects: 10
  Cumulative investment: ₹159,725 crore (~US$18.2B at ~₹88/USD)

--- Projects by state ---
  Gujarat: 4 project(s), ₹124,423 cr
  Odisha: 2 project(s), ₹4,011 cr
  Assam: 1 project(s), ₹27,000 cr
  Andhra Pradesh: 1 project(s), ₹468 cr
  Punjab: 1 project(s), ₹117 cr
  Uttar Pradesh: 1 project(s), ₹3,706 cr

--- Projects by status (as of 2026-05-01) ---
  Approved: 2
  Operational: 2
  Pilot_Operational: 1
  Under_Construction: 5


## Cell 4 — Aggregate to state level + add 4-tier maturity ordinal

**Maturity tiers:**
- **Tier 4 — Operational:** State has ≥1 commercial-production ISM unit (Gujarat only)
- **Tier 3 — Under Construction:** State has ≥1 ISM unit with groundbreaking done but not yet operational
- **Tier 2 — Pre-Construction:** State has ISM-approved unit but no groundbreaking yet
- **Tier 1 — State Policy Only:** State has notified semiconductor/ESDM policy but no ISM-approved unit
- **Tier 0 — No Policy:** All other states

Tier-1 states (state policy + design ecosystem, no ISM manufacturing): Karnataka, Tamil Nadu,
Telangana, Maharashtra, Kerala. Karnataka is particularly notable for the largest GCC concentration
and Lam Research's Feb 2025 manufacturing equipment commitment, but no ISM-approved fab/OSAT.

In [11]:
print("=" * 70)
print("Cell 4 — Aggregate to state level + 4-tier maturity ordinal")
print("=" * 70)

# Aggregate by state
def aggregate_state(group):
    statuses = group['status_2026_05_01'].tolist()
    if 'Operational' in statuses or 'Pilot_Operational' in statuses:
        # Highest-tier wins; Pilot_Operational counted as in-progress operational
        if 'Operational' in statuses:
            tier = 4
        else:
            tier = 3
    elif 'Under_Construction' in statuses:
        tier = 3
    elif 'Approved' in statuses:
        tier = 2
    else:
        tier = 0  # shouldn't happen for ISM-approved

    return pd.Series({
        'facility_count': len(group),
        'total_investment_inr_cr': int(group['investment_inr_cr'].sum()),
        'lead_companies': ';'.join(sorted(set(group['lead_company']))),
        'project_types': ';'.join(sorted(set(group['project_type']))),
        'earliest_approval': group['cabinet_approval_date'].min(),
        'latest_approval': group['cabinet_approval_date'].max(),
        'ism_maturity_tier': tier,
        'pib_prids': ';'.join(sorted(set(group['pib_prid']))),
    })

state_summary = projects_df.groupby('state_name').apply(aggregate_state).reset_index()
state_summary['ism_approved'] = 1

# Add tier-1 states (policy notified but no ISM-approved manufacturing)
tier_1_states = [
    {'state_name': 'Karnataka', 'note': 'Largest design GCC concentration; Lam Research Feb 2025 commitment; no ISM fab/OSAT'},
    {'state_name': 'Tamil Nadu', 'note': 'State policy with ≤50% capex top-up; Foxconn/Samsung assembly; no ISM unit'},
    {'state_name': 'Telangana', 'note': 'Design ecosystem; no ISM-approved manufacturing'},
    {'state_name': 'Maharashtra', 'note': 'Vedanta-Foxconn JV collapsed; data centre policy; no ISM unit'},
    {'state_name': 'Kerala', 'note': 'Design / GCC presence only; no ISM unit'},
]

tier_1_rows = []
for s in tier_1_states:
    tier_1_rows.append({
        'state_name': s['state_name'],
        'facility_count': 0,
        'total_investment_inr_cr': 0,
        'lead_companies': '',
        'project_types': '',
        'earliest_approval': None,
        'latest_approval': None,
        'ism_maturity_tier': 1,
        'pib_prids': '',
        'ism_approved': 0,
        'tier_1_rationale': s['note'],
    })

# Combine ISM-approved (tier 2+) with policy-only (tier 1)
state_summary['tier_1_rationale'] = ''
all_state_summary = pd.concat([state_summary, pd.DataFrame(tier_1_rows)], ignore_index=True)

# Add metadata
all_state_summary['data_source'] = 'NSEMI_Script1c_compiled_from_PIB'
all_state_summary['retrieval_date'] = '2026-05-01'
all_state_summary['analysis_cutoff'] = '2026-05-01'

# Sort: ISM states first (descending tier), then tier-1, then alphabetical within tier
all_state_summary = all_state_summary.sort_values(
    ['ism_approved', 'ism_maturity_tier', 'total_investment_inr_cr', 'state_name'],
    ascending=[False, False, False, True],
).reset_index(drop=True)

out_states = CLEANED / 'rq2_ism_approved_states_n6_verified_2026_05.csv'
all_state_summary.to_csv(out_states, index=False)

print(f"\n✓ Saved: {out_states.name}")
print(f"  Shape: {all_state_summary.shape}")

print(f"\n--- State-level summary ---")
display_cols = ['state_name', 'ism_approved', 'ism_maturity_tier', 'facility_count',
                'total_investment_inr_cr', 'earliest_approval']
print(all_state_summary[display_cols].to_string(index=False))

print(f"\n--- Maturity tier distribution ---")
tier_dist = all_state_summary['ism_maturity_tier'].value_counts().sort_index(ascending=False)
tier_labels = {
    4: 'Operational (commercial production)',
    3: 'Under Construction (groundbreaking done)',
    2: 'Pre-Construction (approved, awaiting groundbreaking)',
    1: 'State Policy Only (no ISM-approved manufacturing)',
}
for tier, count in tier_dist.items():
    label = tier_labels.get(tier, f'Tier {tier}')
    print(f"  Tier {tier} — {label}: {count} state(s)")

Cell 4 — Aggregate to state level + 4-tier maturity ordinal

✓ Saved: rq2_ism_approved_states_n6_verified_2026_05.csv
  Shape: (11, 14)

--- State-level summary ---
    state_name  ism_approved  ism_maturity_tier  facility_count  total_investment_inr_cr earliest_approval
       Gujarat             1                  4               4                   124423        2023-06-15
         Assam             1                  3               1                    27000        2024-02-29
        Odisha             1                  3               2                     4011        2025-08-12
 Uttar Pradesh             1                  3               1                     3706        2025-05-14
Andhra Pradesh             1                  2               1                      468        2025-08-12
        Punjab             1                  2               1                      117        2025-08-12
     Karnataka             0                  1               0                       

## Cell 5 — Final inventory + comparison with existing files

Compares the new files against the existing rq2_pfc_atc_losses.csv and
rq2_semiconductor_facility_locations.csv (read-only, no modification).

**Important:** This notebook deliberately does NOT modify the existing files.
Saturday's RQ2 modeling can choose at runtime which dataset to use:
- Original (n=3 ISM): existing rq2_pfc_atc_losses.csv (interim report alignment)
- Updated (n=6 ISM): join with rq2_ism_approved_states_n6_verified_2026_05.csv (final report)

In [12]:
print("=" * 70)
print("Cell 5 — Final inventory + read-only comparison with existing files")
print("=" * 70)

# Files this notebook produced
new_files = [
    ISM_DIR / 'ism_pib_provenance.json',
    CLEANED / 'rq2_ism_approved_projects_n10_verified_2026_05.csv',
    CLEANED / 'rq2_ism_approved_states_n6_verified_2026_05.csv',
]
new_files += list(ISM_DIR.glob('PIB_PRID_*.pdf')) # Changed to glob for .pdf files

print(f"\n--- Files produced by this notebook ({len(new_files)}) ---")
for f in sorted(new_files):
    if f.exists():
        size_kb = f.stat().st_size / 1024
        rel = f.relative_to(DRIVE_BASE)
        print(f"  ✓ {rel}: {size_kb:.0f} KB")

# Read-only comparison with existing RQ2 files
print(f"\n--- Existing RQ2 files (UNCHANGED, read-only check) ---")
existing_files = [
    ('rq2_pfc_atc_losses.csv', 'AT&C losses outcome variable'),
    ('rq2_semiconductor_facility_locations.csv', 'Original 3-state ISM grouping'),
]
for fname, desc in existing_files:
    p = CLEANED / fname
    if p.exists():
        df_existing = pd.read_csv(p)
        print(f"  ✓ {fname}: {len(df_existing)} rows × {len(df_existing.columns)} cols  ({desc})")
        if 'ism_approved' in df_existing.columns:
            ism_count = df_existing['ism_approved'].sum() if df_existing['ism_approved'].dtype != object else (df_existing['ism_approved'] == 1).sum()
            print(f"    ism_approved=1 rows: {ism_count}")
    else:
        print(f"  ⚠ {fname} NOT FOUND")

# Highlight the 3 NEW ISM states (vs. interim assumption of 3)
print(f"\n--- NEW ISM-approved states discovered (vs. interim n=3 assumption) ---")
projects_df_loaded = pd.read_csv(CLEANED / 'rq2_ism_approved_projects_n10_verified_2026_05.csv')
interim_states = {'Gujarat', 'Assam', 'Uttar Pradesh'}
all_states = set(projects_df_loaded['state_name'].unique())
new_states = sorted(all_states - interim_states)
for s in new_states:
    state_proj = projects_df_loaded[projects_df_loaded['state_name'] == s]
    inv = state_proj['investment_inr_cr'].sum()
    earliest = state_proj['cabinet_approval_date'].min()
    print(f"  + {s}: {len(state_proj)} project(s), ₹{inv:,} cr, earliest approval {earliest}")

print(f"\n--- Summary ---")
print(f"Interim report assumed: 3 ISM-approved states (Gujarat, Assam, Uttar Pradesh)")
print(f"Verified actual count:  6 ISM-approved states")
print(f"  Already in interim:   Gujarat, Assam, Uttar Pradesh")
print(f"  Newly identified:     {', '.join(new_states)}")

print(f"\n--- Recommendation for Saturday's RQ2 modeling ---")
print("OPTION A (binary, n=6): Join cleaned/rq2_pfc_atc_losses.csv with")
print("  cleaned/rq2_ism_approved_states_n6_verified_2026_05.csv on state_name; group by ism_approved.")
print("  Sample: n1=6 ISM, n2=26 non-ISM. Test: Mann-Whitney U + rank-biserial r.")
print()
print("OPTION B (4-tier ordinal): Join cleaned/rq2_pfc_atc_losses.csv with")
print("  cleaned/rq2_ism_approved_states_n6_verified_2026_05.csv on state_name; group by ism_maturity_tier.")
print("  Test: Kruskal-Wallis H + Dunn's post-hoc.")
print()
print("OPTION C (no change): Continue using existing rq2_semiconductor_facility_locations.csv")
print("  with n=3. Document the data cutoff in the limitations section.")
print()
print("--- This notebook produces only NEW files; existing files remain untouched ---")

Cell 5 — Final inventory + read-only comparison with existing files

--- Files produced by this notebook (8) ---
  ✓ cleaned/rq2_ism_approved_projects_n10_verified_2026_05.csv: 3 KB
  ✓ cleaned/rq2_ism_approved_states_n6_verified_2026_05.csv: 2 KB
  ✓ raw/ism_pib_releases/PIB_PRID_1983128.pdf: 116 KB
  ✓ raw/ism_pib_releases/PIB_PRID_2010132.pdf: 151 KB
  ✓ raw/ism_pib_releases/PIB_PRID_2050859.pdf: 122 KB
  ✓ raw/ism_pib_releases/PIB_PRID_2128605.pdf: 119 KB
  ✓ raw/ism_pib_releases/PIB_PRID_2155456.pdf: 132 KB
  ✓ raw/ism_pib_releases/ism_pib_provenance.json: 4 KB

--- Existing RQ2 files (UNCHANGED, read-only check) ---
  ✓ rq2_pfc_atc_losses.csv: 32 rows × 10 cols  (AT&C losses outcome variable)
    ism_approved=1 rows: 3
  ✓ rq2_semiconductor_facility_locations.csv: 7 rows × 6 cols  (Original 3-state ISM grouping)

--- NEW ISM-approved states discovered (vs. interim n=3 assumption) ---
  + Andhra Pradesh: 1 project(s), ₹468 cr, earliest approval 2025-08-12
  + Odisha: 2 project(s),